In [7]:
from pathlib import Path
import os

root = Path.cwd()

while not (root / ".git").exists():
    root = root.parent

os.chdir(root)

print("Project root:", root)

Project root: d:\Projects\btc-research-lab


In [8]:
trade_df = pd.read_parquet(
    root / "data" / "backtests" / "h04c_trades.parquet"
)

print(trade_df.shape)

(2840, 13)


In [9]:
import numpy as np
import pandas as pd

work_df = trade_df.copy()

returns = work_df['net_ret'].values

n_sim = 5000

final_equity = []
max_dd = []

for _ in range(n_sim):

    sim_returns = np.random.permutation(returns)

    equity_curve = np.cumprod(1 + sim_returns)

    running_max = np.maximum.accumulate(equity_curve)

    dd = equity_curve / running_max - 1

    final_equity.append(equity_curve[-1])
    max_dd.append(dd.min())

final_equity = np.array(final_equity)
max_dd = np.array(max_dd)

print("\n===== PHASE 4.1 MONTE CARLO =====")

print(f"\nSimulations: {n_sim}")

print("\n--- Final Equity ---")
print(f"Median       : {np.median(final_equity):.2f}x")
print(f"5% Percentile: {np.percentile(final_equity,5):.2f}x")
print(f"95% Percentile: {np.percentile(final_equity,95):.2f}x")

print("\n--- Drawdown ---")
print(f"Median DD    : {np.median(max_dd):.2%}")
print(f"Worst 5% DD  : {np.percentile(max_dd,5):.2%}")

print("\n--- Risk Metrics ---")
print(f"P(Equity < 1x): {(final_equity < 1).mean():.2%}")
print(f"P(DD > 50%)  : {(max_dd < -0.50).mean():.2%}")


===== PHASE 4.1 MONTE CARLO =====

Simulations: 5000

--- Final Equity ---
Median       : 3.07x
5% Percentile: 3.07x
95% Percentile: 3.07x

--- Drawdown ---
Median DD    : -55.86%
Worst 5% DD  : -71.98%

--- Risk Metrics ---
P(Equity < 1x): 0.00%
P(DD > 50%)  : 75.34%


In [10]:
import numpy as np
import pandas as pd

work_df = trade_df.copy()

returns = work_df['net_ret'].values

n_sim = 5000

final_equity = []
max_dd = []

for _ in range(n_sim):

    sim_returns = np.random.choice(
        returns,
        size=len(returns),
        replace=True
    )

    equity_curve = np.cumprod(1 + sim_returns)

    running_max = np.maximum.accumulate(equity_curve)

    dd = equity_curve / running_max - 1

    final_equity.append(equity_curve[-1])
    max_dd.append(dd.min())

final_equity = np.array(final_equity)
max_dd = np.array(max_dd)

print("\n===== BOOTSTRAP MONTE CARLO =====")

print("\n--- Final Equity ---")
print(f"Median       : {np.median(final_equity):.2f}x")
print(f"5% Percentile: {np.percentile(final_equity,5):.2f}x")
print(f"95% Percentile: {np.percentile(final_equity,95):.2f}x")

print("\n--- Drawdown ---")
print(f"Median DD    : {np.median(max_dd):.2%}")
print(f"Worst 5% DD  : {np.percentile(max_dd,5):.2%}")

print("\n--- Risk ---")
print(f"P(Equity < 1x): {(final_equity < 1).mean():.2%}")
print(f"P(DD > 50%)  : {(max_dd < -0.50).mean():.2%}")


===== BOOTSTRAP MONTE CARLO =====

--- Final Equity ---
Median       : 3.10x
5% Percentile: 0.58x
95% Percentile: 16.34x

--- Drawdown ---
Median DD    : -55.93%
Worst 5% DD  : -80.14%

--- Risk ---
P(Equity < 1x): 13.32%
P(DD > 50%)  : 66.60%


In [11]:
import pandas as pd
import numpy as np

work_df = trade_df.copy()

summary = []

years = sorted(work_df['year'].unique())

for split_year in years[:-1]:

    train = work_df[work_df['year'] <= split_year]
    test = work_df[work_df['year'] > split_year]

    train_pf = (
        train.loc[train.net_ret > 0, 'net_ret'].sum()
        /
        abs(train.loc[train.net_ret < 0, 'net_ret'].sum())
    )

    test_pf = (
        test.loc[test.net_ret > 0, 'net_ret'].sum()
        /
        abs(test.loc[test.net_ret < 0, 'net_ret'].sum())
    )

    summary.append({
        'train_end': split_year,
        'train_trades': len(train),
        'test_trades': len(test),
        'train_pf': train_pf,
        'test_pf': test_pf,
        'train_hit': (train.net_ret > 0).mean(),
        'test_hit': (test.net_ret > 0).mean(),
        'train_mean': train.net_ret.mean(),
        'test_mean': test.net_ret.mean()
    })

wf = pd.DataFrame(summary)

print("\n===== WALK FORWARD VALIDATION =====")
print(wf)

print("\nAverage OOS PF :", wf['test_pf'].mean())
print("Worst OOS PF   :", wf['test_pf'].min())
print("OOS PF > 1     :", (wf['test_pf'] > 1).sum(), "/", len(wf))


===== WALK FORWARD VALIDATION =====
   train_end  train_trades  test_trades  train_pf   test_pf  train_hit  \
0       2019           428         2412  1.148531  1.088208   0.549065   
1       2020           848         1992  1.198249  1.055735   0.568396   
2       2021          1734         1106  1.142293  1.018599   0.555363   
3       2022          2262          578  1.084721  1.159197   0.545093   
4       2023          2474          366  1.103889  1.047246   0.550121   

   test_hit  train_mean  test_mean  
0  0.549337    0.000865   0.000528  
1  0.541165    0.001148   0.000336  
2  0.539783    0.000881   0.000104  
3  0.565744    0.000529   0.000774  
4  0.543716    0.000625   0.000262  

Average OOS PF : 1.073797001437781
Worst OOS PF   : 1.0185986268449543
OOS PF > 1     : 5 / 5


In [12]:
import numpy as np
import pandas as pd

work_df = trade_df.copy()

sizes = [0.1, 0.2, 0.3, 0.5, 1.0]

summary = []

for size in sizes:

    equity = np.cumprod(
        1 + size * work_df['net_ret']
    )

    running_max = np.maximum.accumulate(equity)

    dd = equity / running_max - 1

    cagr_proxy = equity[-1] ** (1/6) - 1

    summary.append({
        'size': size,
        'final_equity': equity[-1],
        'max_dd': dd.min(),
        'cagr_proxy': cagr_proxy
    })

summary_df = pd.DataFrame(summary)

print("\n===== POSITION SIZE SWEEP =====")
print(summary_df)

KeyError: -1

In [13]:
import pandas as pd
import numpy as np

work_df = trade_df.copy()

work_df = work_df.sort_values('entry_time')

equity = (1 + work_df['net_ret']).cumprod()

running_max = equity.cummax()

drawdown = equity / running_max - 1

work_df['drawdown'] = drawdown

worst_idx = drawdown.idxmin()

print("\n===== DRAWDOWN ANATOMY =====")

print(f"Worst DD : {drawdown.min():.2%}")

print("\nWorst Trade:")
print(work_df.loc[worst_idx])

print("\nTrades inside DD > 40%:")
print((drawdown < -0.40).sum())

print("\nTrades inside DD > 50%:")
print((drawdown < -0.50).sum())

print("\nYear Distribution:")
print(
    work_df.loc[
        drawdown < -0.40,
        'year'
    ].value_counts().sort_index()
)


===== DRAWDOWN ANATOMY =====
Worst DD : -51.26%

Worst Trade:
entry_time     2020-03-12 23:10:00
exit_time      2020-03-13 02:10:00
entry_price                5520.69
exit_price                 3882.22
gross_ret                -0.296787
net_ret                  -0.297187
duration                        36
year                          2020
equity                    0.719397
vol_regime                    High
bull                         False
mae                      -0.302623
mfe                       0.024928
drawdown                 -0.512588
Name: 488, dtype: object

Trades inside DD > 40%:
132

Trades inside DD > 50%:
1

Year Distribution:
year
2020     20
2022    111
2023      1
Name: count, dtype: int64


In [14]:
import pandas as pd
import numpy as np

work_df = trade_df.copy()

thresholds = [-0.05, -0.10, -0.15, -0.20]

print("\n===== TAIL LOSS ANALYSIS =====")

total_loss = abs(
    work_df.loc[
        work_df.net_ret < 0,
        'net_ret'
    ].sum()
)

for t in thresholds:

    subset = work_df[
        work_df.net_ret <= t
    ]

    loss_contrib = abs(
        subset['net_ret'].sum()
    )

    print(f"\nLoss <= {t:.0%}")

    print(f"Trades : {len(subset)}")

    print(
        f"Loss Share : "
        f"{loss_contrib/total_loss:.2%}"
    )

    if len(subset) > 0:
        print(
            f"Mean Loss : "
            f"{subset.net_ret.mean():.2%}"
        )


===== TAIL LOSS ANALYSIS =====

Loss <= -5%
Trades : 26
Loss Share : 11.94%
Mean Loss : -7.77%

Loss <= -10%
Trades : 2
Loss Share : 2.78%
Mean Loss : -23.50%

Loss <= -15%
Trades : 2
Loss Share : 2.78%
Mean Loss : -23.50%

Loss <= -20%
Trades : 1
Loss Share : 1.76%
Mean Loss : -29.72%


In [15]:
import pandas as pd
import numpy as np

work_df = trade_df.copy()

work_df = work_df.sort_values('entry_time')

losses = (work_df['net_ret'] < 0).astype(int)

max_streak = 0
current = 0
streaks = []

for x in losses:

    if x == 1:
        current += 1
        max_streak = max(max_streak, current)
    else:
        if current > 0:
            streaks.append(current)
        current = 0

if current > 0:
    streaks.append(current)

print("\n===== LOSING STREAK ANALYSIS =====")

print("Trades:", len(work_df))
print("Loss Rate:", losses.mean())

print("\nMax Losing Streak:", max_streak)

print("Median Losing Streak:",
      np.median(streaks))

print("95% Losing Streak:",
      np.percentile(streaks, 95))

print("Number of Streaks:",
      len(streaks))


===== LOSING STREAK ANALYSIS =====
Trades: 2840
Loss Rate: 0.4507042253521127

Max Losing Streak: 11
Median Losing Streak: 1.0
95% Losing Streak: 4.0
Number of Streaks: 710


In [16]:
import numpy as np
import pandas as pd

work_df = trade_df.copy()

sizes = [0.05, 0.10, 0.20, 0.30, 0.50, 1.00]

summary = []

for size in sizes:

    equity = (1 + size * work_df['net_ret']).cumprod()

    running_max = equity.cummax()

    dd = equity / running_max - 1

    summary.append({
        'size': size,
        'final_equity': float(equity.iloc[-1]),
        'max_dd': float(dd.min())
    })

summary_df = pd.DataFrame(summary)

print("\n===== POSITION SIZE SWEEP =====")
print(summary_df)


===== POSITION SIZE SWEEP =====
   size  final_equity    max_dd
0  0.05      1.084236 -0.030630
1  0.10      1.172564 -0.060643
2  0.20      1.360873 -0.118845
3  0.30      1.563202 -0.174660
4  0.50      1.999085 -0.279337
5  1.00      3.066908 -0.512588


In [17]:
import numpy as np
import pandas as pd

work_df = trade_df.copy()

r = work_df['net_ret']

win_rate = (r > 0).mean()

avg_win = r[r > 0].mean()

avg_loss = abs(r[r < 0].mean())

b = avg_win / avg_loss

kelly = win_rate - ((1 - win_rate) / b)

print("\n===== KELLY ESTIMATE =====")

print(f"Win Rate  : {win_rate:.4f}")
print(f"Avg Win   : {avg_win:.4%}")
print(f"Avg Loss  : {avg_loss:.4%}")
print(f"Payoff    : {b:.4f}")

print(f"\nKelly Fraction : {kelly:.2%}")

print(f"Half Kelly     : {kelly/2:.2%}")
print(f"Quarter Kelly  : {kelly/4:.2%}")


===== KELLY ESTIMATE =====
Win Rate  : 0.5493
Avg Win   : 1.1901%
Avg Loss  : 1.3221%
Payoff    : 0.9002

Kelly Fraction : 4.86%
Half Kelly     : 2.43%
Quarter Kelly  : 1.22%


In [18]:
import numpy as np
import pandas as pd

work_df = trade_df.copy()

returns = work_df['net_ret'].values

kelly = 0.0486

sizes = {
    'QuarterKelly': kelly/4,
    'HalfKelly': kelly/2,
    'FullKelly': kelly,
    'DoubleKelly': kelly*2
}

n_sim = 3000

results = []

for label, size in sizes.items():

    ruin_count = 0

    dd30 = 0
    dd50 = 0

    for _ in range(n_sim):

        sim = np.random.choice(
            returns,
            size=len(returns),
            replace=True
        )

        equity = np.cumprod(
            1 + size * sim
        )

        running_max = np.maximum.accumulate(equity)

        dd = equity / running_max - 1

        mdd = dd.min()

        if equity[-1] < 1:
            ruin_count += 1

        if mdd < -0.30:
            dd30 += 1

        if mdd < -0.50:
            dd50 += 1

    results.append({
        'size': label,
        'ruin_prob': ruin_count / n_sim,
        'dd30_prob': dd30 / n_sim,
        'dd50_prob': dd50 / n_sim
    })

print(pd.DataFrame(results))

           size  ruin_prob  dd30_prob  dd50_prob
0  QuarterKelly   0.060000        0.0        0.0
1     HalfKelly   0.055333        0.0        0.0
2     FullKelly   0.056667        0.0        0.0
3   DoubleKelly   0.060333        0.0        0.0


In [19]:
from pathlib import Path

report_dir = root / "reports"
report_dir.mkdir(exist_ok=True)

report_text = """
# Phase 4 — Validation Report

## Objective

Determine whether H04c represents a deployable trading edge
or a statistical artifact.

---

## Walk-Forward Validation

Results:

2019 -> PF 1.088
2020 -> PF 1.056
2021 -> PF 1.019
2022 -> PF 1.159
2023 -> PF 1.047

Summary:

- OOS PF > 1 in all windows (5/5)
- Worst OOS PF = 1.019

Conclusion:

Edge survives out-of-sample testing.

---

## Bootstrap Monte Carlo

Median Equity : 3.10x

5% Percentile : 0.58x

95% Percentile : 16.34x

P(Equity < 1x) : 13.3%

Conclusion:

Edge appears genuine but relatively weak.

---

## Drawdown Analysis

Worst Historical DD:

-51.3%

Largest Loss:

2020-03-12
Return = -29.7%

Observation:

Drawdowns are concentrated in specific market regimes,
especially March 2020 and the 2022 bear market.

---

## Tail Risk Analysis

Loss <= -5% :

26 trades

11.9% of total losses

Loss <= -10% :

2 trades

Conclusion:

Tail losses are not the primary source of drawdown.

---

## Losing Streak Analysis

Max Losing Streak : 11

Median Losing Streak : 1

95th Percentile : 4

Conclusion:

Extreme losing streaks are not responsible for DD.

---

## Position Sizing

5% Allocation:

Max DD ≈ -3%

10% Allocation:

Max DD ≈ -6%

20% Allocation:

Max DD ≈ -12%

30% Allocation:

Max DD ≈ -17%

Conclusion:

Most drawdown concerns can be mitigated through
conservative position sizing.

---

## Kelly Estimate

Kelly Fraction : 4.86%

Half Kelly : 2.43%

Quarter Kelly : 1.22%

Conclusion:

Optimal sizing is small and consistent with
observed drawdown behaviour.

---

## Final Assessment

H04c survives:

- Transaction Costs
- Walk-Forward Validation
- Bootstrap Monte Carlo

The strategy exhibits:

✓ Persistent out-of-sample edge

✓ Parameter stability

✓ Regime dependence

✓ Practical tradability

Primary weakness:

PF ≈ 1.10

Current status:

VALIDATED EDGE

Suitable for Deployment Research.

END OF PHASE 4 REPORT
"""

report_path = report_dir / "phase4_validation_report.md"

with open(report_path, "w", encoding="utf-8") as f:
    f.write(report_text)

print(report_path)

d:\Projects\btc-research-lab\reports\phase4_validation_report.md


In [20]:
from pathlib import Path
import pandas as pd

validation_dir = root / "data" / "validation"
validation_dir.mkdir(parents=True, exist_ok=True)

# Position sizing results
position_df = pd.DataFrame({
    "size": [0.05, 0.10, 0.20, 0.30, 0.50, 1.00],
    "final_equity": [1.084236, 1.172564, 1.360873, 1.563202, 1.999085, 3.066908],
    "max_dd": [-0.030630, -0.060643, -0.118845, -0.174660, -0.279337, -0.512588]
})

position_df.to_csv(
    validation_dir / "phase4_position_sizing.csv",
    index=False
)

# Walk-forward results
wf = pd.DataFrame({
    "train_end":[2019,2020,2021,2022,2023],
    "test_pf":[1.088208,1.055735,1.018599,1.159197,1.047246]
})

wf.to_csv(
    validation_dir / "phase4_walkforward.csv",
    index=False
)

print("Validation outputs saved.")

Validation outputs saved.
